# Kaggle Homework 2 — Tyler Wolf Williams (tylerwolfwilliams2)
**Competition:** [Playground Series S6E4 — Irrigation Need Prediction](https://www.kaggle.com/competitions/playground-series-s6e4)  
**Deadline:** April 30, 2026  
**Metric:** Accuracy (multi-class: Low / Medium / High)

---

## Notebook Links

| Notebook | Purpose |
|---|---|
| `HW2_FeatureEngineering_Tyler.ipynb` | Feature engineering, SHAP analysis, leakage investigation |
| `HW2_NewModels_Ensemble_Tyler.ipynb` | MLP, HistGBM, DART, stacking ensemble, Kaggle submissions |

*(Run feature engineering notebook first — models notebook loads its output.)*

---

## Dataset Overview

- **Train:** 630,000 rows × 21 columns  
- **Test:** 270,000 rows × 20 columns  
- **Target (class distribution):** Low 58.7% · Medium 38.0% · High 3.3%  
- **Key challenge:** 'High' class is severely underrepresented (~1/18th of 'Low'); most models default to predicting Low/Medium and miss High class recall

---

## Feature Engineering Summary

### Motivation
The raw dataset has 19 features — 11 numerical, 8 categorical. Phase 1 & 2 used zero feature engineering. For HW2, I designed features grounded in **agricultural domain knowledge** about what drives irrigation need.

### Features Created

**Numerical (14 new):**

| Feature | Formula | Rationale |
|---|---|---|
| `log_Rainfall_mm` | log(1 + Rainfall) | Compress right skew; better tree splits |
| `log_Wind_Speed_kmh` | log(1 + Wind) | Same — wind is heavy-tailed |
| `log_Field_Area_hectare` | log(1 + Area) | Field sizes vary over orders of magnitude |
| `log_Previous_Irrigation_mm` | log(1 + Prev_Irr) | Previous irrigation is zero-inflated and skewed |
| `ET_proxy` | (Temp × Sunlight × Wind) / (Humidity + 1) | Evapotranspiration demand — core driver of irrigation need (simplified Penman-Monteith) |
| `water_balance` | Rainfall + Soil_Moisture − ET_proxy | Net water deficit/surplus; negative = irrigation needed |
| `aridity_index` | Rainfall / (Temp + 1) | Lower values = drier climate needing more irrigation |
| `pH_deviation` | |Soil_pH − 6.5| | Deviation from optimal pH — stressed soil |
| `soil_quality` | OC / (EC + 0.1) − pH_dev | Composite: organic carbon retention minus salinity+pH stress |
| `moisture_x_temp` | Soil_Moisture × Temp | Interaction: hot-dry conditions need irrigation most |
| `rain_x_wind` | Rainfall × Wind | Wind-driven evaporation effect |
| `pH_x_EC` | pH × EC | Soil chemistry interaction |
| `humidity_x_temp` | Humidity × Temp | Apparent humidity / heat index proxy |
| `prev_irr_per_ha` | Prev_Irr / (Area + 0.01) | Irrigation intensity normalised by field size |

**Categorical cross-features (3 new):**
- `crop_stage` = Crop_Type + Crop_Growth_Stage (captures stage-specific water needs)
- `season_region` = Season + Region (regional climate × seasonal patterns)
- `soil_crop` = Soil_Type + Crop_Type (soil-crop compatibility)

**Target encoding (7 features, 5-fold CV to prevent leakage):**  
Crop_Type, Soil_Type, Season, Region, crop_stage, season_region, soil_crop

**Group statistics (10 features):**  
Mean/std of Soil_Moisture by Crop_Type and Region; Temp by Season; Rainfall by Region; ET_proxy by Crop_Type

### Leakage Investigation
`Irrigation_Type` and `Water_Source` may be downstream of the target (the irrigation system is *chosen after* the need is determined). I tested model performance with vs. without these features:

| Feature Set | CV Accuracy | CV F1 Macro |
|---|---|---|
| Base 19 features | *(fill after run)* | *(fill after run)* |
| All engineered (incl. leakage cols) | *(fill after run)* | *(fill after run)* |
| All engineered (excl. leakage cols) | *(fill after run)* | *(fill after run)* |
| Selected (top SHAP) | *(fill after run)* | *(fill after run)* |

### Feature Selection (SHAP)
Features with mean |SHAP| below the 25th percentile were dropped. The final feature set has **[fill after run]** features.

**Top SHAP features (expected):** Soil_Moisture, ET_proxy, water_balance, moisture_x_temp, Crop_Growth_Stage, Rainfall_mm, Temperature_C

---

## Models — What I Tried

### Phase 1 & 2 Recap
- **Random Forest** (Phase 1): CV Acc 0.9855, F1 0.9710, Kaggle LB 0.95876
- **LightGBM gbdt** (Phase 1): CV Acc 0.9847, F1 0.9701, Kaggle LB 0.95937 ← previous best
- **XGBoost** (Phase 2): CV Acc 0.9846, F1 0.9700, Kaggle LB 0.95875
- **CatBoost** (Phase 2): CV Acc 0.9839, F1 0.9686, Kaggle LB 0.95615

### HW2 New Models

#### MLP Neural Network
**Why different:** Gradient descent through continuous weights, not greedy splits. Learns shared hidden representations across classes. Sensitive to feature scale (uses StandardScaler). Can capture smooth, non-axis-aligned decision boundaries that tree models miss.

**Tuning approach:** RandomizedSearchCV over architecture, regularisation (alpha), learning rate, and activation function. Tuned on a 150K stratified sample for speed; final evaluation on full data.

**Results:**
- Baseline CV F1: *(fill after run)*
- Tuned CV F1: *(fill after run)*
- Kaggle LB: *(fill after submission)*

**Observation:** MLP was [faster/slower] than expected on 630K rows. The architecture [(256,128,64) etc.] worked best. Performance was [above/below] tree models on this tabular dataset, which is typical — tree models generally edge out MLPs on structured/tabular data.

#### HistGradientBoostingClassifier
**Why different:** sklearn's own GBM implementation — completely independent from LightGBM/XGBoost codebases. Uses fixed-bin histograms (255 bins), different regularisation via `l2_regularization` and `min_samples_leaf`, and supports native `class_weight='balanced'` in the objective. Deterministic across threads.

**Tuning approach:** RandomizedSearchCV (20 iterations, 5-fold) over max_iter, learning_rate, max_depth, min_samples_leaf, l2_regularization, max_leaf_nodes.

**Results:**
- Baseline CV F1: *(fill after run)*
- Tuned CV F1: *(fill after run)*
- Kaggle LB: *(fill after submission)*

**Observation:** HistGBM was [competitive/outperformed/underperformed] relative to LightGBM. The key difference was [X parameter] — *(fill in after seeing results)*.

#### LightGBM DART (Dropout Boosting)
**Why different:** Same library as Phase 1 LightGBM but an entirely different boosting algorithm. DART applies neural-network-style dropout: at each iteration, a random fraction of existing trees is dropped (excluded) before fitting the next tree. This prevents any early tree from dominating the ensemble and acts as strong regularisation. Key DART-specific parameters: `drop_rate` (fraction of trees dropped) and `skip_drop` (probability of skipping dropout entirely).

**Tuning approach:** RandomizedSearchCV (15 iterations, 5-fold) over num_leaves, max_depth, drop_rate, skip_drop, reg_alpha, reg_lambda.

**Results:**
- Baseline CV F1: *(fill after run)*
- Tuned CV F1: *(fill after run)*
- Kaggle LB: *(fill after submission)*

**Observation:** DART [helped/hurt/matched] compared to standard gbdt. The `drop_rate` of [X] worked best. Minority class ('High') recall [improved/did not improve], suggesting dropout regularisation [did/did not] help with class imbalance.

---

## Ensemble Approaches

### Stacking (Meta-Learner)
**Method:** 3-fold out-of-fold probability predictions from HGB, DART, and MLP → Logistic Regression meta-learner. The meta-learner sees 9 meta-features (3 models × 3 class probabilities) and learns optimal weights.

**Results (holdout 20%):**
- Accuracy: *(fill after run)*
- F1 Macro: *(fill after run)*
- Kaggle LB: *(fill after submission)*

### Probability Averaging
**Equal-weight (33/33/33%):** Averages probability outputs from all three models equally.

**Custom-weight (40/40/20%):** Upweights tree models (HGB and DART) relative to MLP, reflecting tree models' typical advantage on tabular data.

---

## Full Performance Table

| Model | Phase | CV/Holdout Acc | CV/Holdout F1 | Kaggle LB |
|---|---|---|---|---|
| Random Forest | 1 | 0.9855 | 0.9710 | 0.95876 |
| LightGBM gbdt | 1 | 0.9847 | 0.9701 | **0.95937** |
| XGBoost | 2 | 0.9846 | 0.9700 | 0.95875 |
| CatBoost | 2 | 0.9839 | 0.9686 | 0.95615 |
| MLP (Baseline) | 2 | *(run)* | *(run)* | — |
| MLP (Tuned) | 2 | *(run)* | *(run)* | *(submit)* |
| HistGBM (Baseline) | 2 | *(run)* | *(run)* | — |
| HistGBM (Tuned) | 2 | *(run)* | *(run)* | *(submit)* |
| DART (Baseline) | 2 | *(run)* | *(run)* | — |
| DART (Tuned) | 2 | *(run)* | *(run)* | *(submit)* |
| Stacking Ensemble | 2 | *(run)* | *(run)* | *(submit)* |
| Avg Ensemble — Equal | 2 | *(run)* | *(run)* | *(submit)* |
| Avg Ensemble — Weighted | 2 | *(run)* | *(run)* | *(submit)* |

---

## Discussion

### What Worked Well
- **Evapotranspiration proxy (`ET_proxy`):** Directly captures the physical driver of irrigation need. Expected to be a top-3 SHAP feature.
- **Water balance feature:** Net deficit/surplus combines multiple raw features into a single interpretable signal.
- **Target encoding of `crop_stage`:** The interaction of Crop_Type and Crop_Growth_Stage captures stage-specific water needs that neither feature captures alone.
- **class_weight='balanced':** Essential for the 3.3% 'High' class — without it, all tree models underpredict High. Applied consistently across all HW2 models.

### What Didn't Work Well
- **`crop_season_aligned`:** Crop names in the dataset may be synthetic and not match real Indian agricultural crop names (Kharif/Rabi/Zaid alignment). If SHAP ranked it near zero, this feature failed as designed.
- **CatBoost (Phase 2):** Showed the worst Kaggle LB (0.95615) despite similar CV scores — likely a worse fit to this dataset's structure than leaf-wise models.

### Performance Improvements
- Feature engineering vs. base features: **+[X] F1 points** — *(fill after run)*
- Stacking vs. best single model: **+[X] F1 / −[X] F1** — *(fill after run)*
- Whether improvements were meaningful or marginal: given CV scores are already at 0.98+, marginal gains are expected. The main value is in minority class ('High') recall.

### Moving Forward
- **Keep:** ET_proxy, water_balance, target encoding, class_weight throughout. HistGBM if competitive.
- **Reconsider:** MLP may not add enough above tree models to justify training time. Revisit with a smaller architecture.
- **Future exploration:** Threshold tuning to improve 'High' class recall; SMOTE oversampling for training only; feature selection using permutation importance as a second opinion to SHAP.

---

## Leaderboard History

| Submission | Date | Public LB | Notes |
|---|---|---|---|
| Random Forest (Phase 1) | Apr 2026 | 0.95876 | Baseline bagging model |
| LightGBM gbdt (Phase 1) | Apr 2026 | 0.95937 | Best Phase 1 |
| XGBoost (Phase 2) | Apr 2026 | 0.95875 | |
| CatBoost (Phase 2) | Apr 2026 | 0.95615 | Worst — symmetric tree architecture |
| MLP Tuned (HW2) | — | TBD | |
| HistGBM Tuned (HW2) | — | TBD | |
| DART Tuned (HW2) | — | TBD | |
| Stacking Ensemble (HW2) | — | TBD | |
| Avg Ensemble (HW2) | — | TBD | |